In [ ]:
import numpy as np 
from pathlib import Path

from ase.io import read, write
from ase import Atoms, units

[AENET](http://ann.atomistic.net/documentation/#structural-energy-reference-data) uses a specific XSF file format, that contains the total energy in its heading:  



```xsf
# total energy = -4990.44928342 eV

CRYSTAL
PRIMVEC
   2.967  0.000  0.000
   0.000  4.648  0.000
   0.000 -0.000  4.648
PRIMCOORD
6 1
Ti 1.483  2.324  2.324  0.000  0.000  0.000
Ti 0.000  0.000  0.000  0.000  0.000  0.000
O  1.483  0.905  0.905  0.000 -0.004 -0.004
O  1.483  3.742  3.742  0.000  0.004  0.004
O  0.000  1.418  3.230  0.000  0.004 -0.004
O  0.000  3.230  1.418  0.000 -0.004  0.004
```

In [ ]:
# ZnO_example = read('ZnO-3.14-1.55-212-0.06-2.out', format='espresso-out')
# print(ZnO_example)
# print(f'Energia em Ry:\n{ZnO_example.get_potential_energy() / units.Ry}')
# # ZnO_example.write('ZnO-3.14-1.55-212-0.06-2.xsf', format='xsf')
# # ZnO_example
# # path_out_QE =  'ZnO-3.14-1.55-212-0.06-2.out'
# # Path(path_out_QE).with_suffix('.xsf')
# # ZnO_example

In [ ]:
def qe2xsf_aenet(scf_output: str | Path, write: bool = False) -> str:
    scf_output = Path(scf_output)
    atoms: Atoms = read(scf_output, format='espresso-out')
    
    energy = atoms.get_potential_energy()
    forces = atoms.get_forces()
    
    xsf_filename = Path(scf_output).with_suffix('.xsf') # .out -> .xsf

    xsf: list[str] = ['# total energy = {} eV'.format(energy), '']
    
    if True in atoms.pbc:
        # CRYSTAL and PRIMVEC sections
        xsf += ['CRYSTAL', 'PRIMVEC']
        
        # Unit Cell Vectors
        for v in atoms.get_cell():
            xsf += ['{} {} {}'.format(*v)]

        # Atomic Positions and Forces Header
        xsf += ['PRIMCOORD', '{} 1'.format(len(atoms))]

    else:
        xsf += ['ATOMS']
    
    atom_line = ('{atom.symbol:<3s} {atom.x: .12f} {atom.y: .12f} {atom.z: .12f}'
         ' {f[0]: .12f} {f[1]: .12f} {f[2]: .12f}')
    # Append Atomic Positions and Forces after Header
    xsf += [atom_line.format(atom=atom, f=forces[i]) for i, atom in enumerate(atoms)]
    # Concatenate all lines.
    output: str = '\n'.join(xsf)

    # Write the ASE XSF file:
    if write:
        with open(xsf_filename, 'w') as f:
            f.write(output)
    
    
    return output

In [ ]:
qe2xsf_aenet('ZnO-3.14-1.55-212-0.06-2.out', write=True)